# Step 3 — Statistical Analysis

Correlation analysis, Chi-Square test for categorical features, and Hypothesis testing for numerical features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../data/processed/cleaned_data.csv', index_col=0)
print('Shape:', df.shape)

## 3.1 Correlation Analysis

In [ ]:
num_cols = ['Age', 'Job', 'Credit amount', 'Duration', 'Risk']
corr = df[num_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix (Numeric Features)')
plt.tight_layout()
plt.show()

print('\nCorrelation with Risk (sorted):')
print(corr['Risk'].sort_values(ascending=False).to_string())

## 3.2 Chi-Square Test

**H0:** The categorical feature is independent of Risk.  
**H1:** The categorical feature is associated with Risk.  
Reject H0 if p-value < 0.05.

In [ ]:
cat_features = ['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']
chi2_results = []

for col in cat_features:
    ct = pd.crosstab(df[col], df['Risk'])
    chi2, p, dof, expected = chi2_contingency(ct)
    chi2_results.append({
        'Feature': col,
        'Chi2 Stat': round(chi2, 3),
        'p-value': round(p, 4),
        'DOF': dof,
        'Significant (p<0.05)': '✓ Yes' if p < 0.05 else '✗ No'
    })

chi2_df = pd.DataFrame(chi2_results)
print('=== Chi-Square Test Results ===')
print(chi2_df.to_string(index=False))

In [ ]:
# Visualize Chi2 statistics
plt.figure(figsize=(8, 4))
plt.barh(chi2_df['Feature'], chi2_df['Chi2 Stat'], color='steelblue', edgecolor='black')
plt.axvline(x=3.84, color='red', linestyle='--', label='Critical value (p=0.05, dof=1)')
plt.title('Chi-Square Statistics by Feature')
plt.xlabel('Chi2 Statistic')
plt.legend()
plt.tight_layout()
plt.show()

## 3.3 Hypothesis Testing (Mann-Whitney U Test)

**H0:** No difference in distribution between Good and Bad risk groups.  
**H1:** Significant difference exists.  
Using Mann-Whitney U (non-parametric, no normality assumption required).

In [ ]:
good = df[df['Risk'] == 0]
bad  = df[df['Risk'] == 1]

hyp_results = []
for col in ['Age', 'Credit amount', 'Duration']:
    stat, p = mannwhitneyu(good[col], bad[col], alternative='two-sided')
    hyp_results.append({
        'Feature': col,
        'Good Mean': round(good[col].mean(), 2),
        'Bad Mean':  round(bad[col].mean(), 2),
        'Mean Diff': round(bad[col].mean() - good[col].mean(), 2),
        'U-Stat': round(stat, 2),
        'p-value': round(p, 4),
        'Reject H0 (p<0.05)': '✓ Yes' if p < 0.05 else '✗ No'
    })

hyp_df = pd.DataFrame(hyp_results)
print('=== Mann-Whitney U Test Results ===')
print(hyp_df.to_string(index=False))

In [ ]:
# Visualize mean comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['Age', 'Credit amount', 'Duration']):
    ax.hist(good[col], bins=25, alpha=0.6, label='Good (0)', color='steelblue', edgecolor='black')
    ax.hist(bad[col],  bins=25, alpha=0.6, label='Bad (1)',  color='tomato',    edgecolor='black')
    ax.set_title(col)
    ax.legend()
plt.suptitle('Feature Distributions: Good vs Bad Risk')
plt.tight_layout()
plt.show()